In [ ]:
from google.colab import drive
drive.mount('/content/drive')

"/content/drive/MyDrive/" --> drive path

In [ ]:
import os

base_path = "/content/drive/MyDrive/ADNI1"

print(os.listdir(base_path))


to check the shape of the nii file

In [ ]:
import nibabel as nib
img = nib.load("/content/drive/MyDrive/ADNI1/ADNI1_CN/ADNI_002_S_0413.nii")
print(img.get_fdata().shape)


In [ ]:
import os

# Correct base path
base_path = "/content/drive/MyDrive/ADNI1"

# Define classes as DICTIONARY (NOT list)
classes = {
    "AD": "ADNI1_AD",
    "CN": "ADNI1_CN",
    "MCI": "ADNI1_MCI"
}


In [ ]:
for label, folder in classes.items():
    path = os.path.join(base_path, folder)
    print(label, ":", len(os.listdir(path)))

In [ ]:
import random

split_data = {}

for label, folder in classes.items():
    files = os.listdir(os.path.join(base_path, folder))
    random.shuffle(files)

    train_split = int(0.7 * len(files))
    val_split = int(0.85 * len(files))

    split_data[label] = {
        "train": files[:train_split],
        "val": files[train_split:val_split],
        "test": files[val_split:]
    }

    print("\n", label)
    print("Train:", len(split_data[label]["train"]))
    print("Val:", len(split_data[label]["val"]))
    print("Test:", len(split_data[label]["test"]))


In [ ]:
import nibabel as nib
import numpy as np
import cv2

output_base = "/content/drive/MyDrive/ADNI1/processed"

def save_slices(file_list, label, split_type):

    for file in file_list:
        file_path = os.path.join(base_path, classes[label], file)
        nii = nib.load(file_path)
        img = nii.get_fdata()

        # Middle 60 slices
        start = img.shape[2]//2 - 30
        end = img.shape[2]//2 + 30

        for i in range(start, end):
            slice_img = img[:, :, i]

            # Normalize
            slice_img = (slice_img - np.min(slice_img)) / (np.max(slice_img) - np.min(slice_img) + 1e-8)
            slice_img = (slice_img * 255).astype(np.uint8)

            # Resize
            slice_img = cv2.resize(slice_img, (160,160))

            save_folder = os.path.join(output_base, split_type, label)
            os.makedirs(save_folder, exist_ok=True)

            cv2.imwrite(os.path.join(save_folder, f"{file}_{i}.png"), slice_img)


In [ ]:
for label in classes.keys():
    for split_type in ["train", "val", "test"]:
        print("Processing:", label, split_type)
        save_slices(split_data[label][split_type], label, split_type)

In [ ]:
import os

base_processed = "/content/drive/MyDrive/ADNI1/processed"

for split in ["train", "val", "test"]:
    total = 0
    print("\n", split)
    for cls in ["AD", "CN", "MCI"]:
        count = len(os.listdir(os.path.join(base_processed, split, cls)))
        print(cls, ":", count)
        total += count
    print("Total:", total)


In [ ]:
import os
base = "/content/drive/MyDrive/ADNI1/processed"
for split in ["train", "val", "test"]:
    print(f"\n{split.upper()}")
    for cls in ["AD", "CN", "MCI"]:
        folder = os.path.join(base, split, cls)
        print(f"  {cls}: {len(os.listdir(folder))}")

### Model 1: Simple CNN (replicates paper baseline)
### Model 2: ResNet50 Transfer Learning (proposed improvement)

Cell 1: Install & Import

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

Cell 2: Paths & Config

In [ ]:
# Mount Google Drive first:
# from google.colab import drive
# drive.mount('/content/drive')
BASE_PATH  = "/content/processed"
TRAIN_DIR  = "/content/processed/train"
VAL_DIR    = "/content/processed/val"
TEST_DIR   = "/content/processed/test"
IMG_SIZE   = (160, 160)
BATCH_SIZE = 32
EPOCHS_CNN = 30
EPOCHS_RES = 20
CLASSES    = ["AD", "CN", "MCI"]
NUM_CLASSES = 3

TRAIN_DIR = os.path.join(BASE_PATH, "train")
VAL_DIR   = os.path.join(BASE_PATH, "val")
TEST_DIR  = os.path.join(BASE_PATH, "test")

Cell 3: Print Dataset Counts

In [ ]:
print("Dataset counts:")
for split in ["train", "val", "test"]:
    print(f"\n{split.upper()}")
    total = 0
    for cls in CLASSES:
        count = len(os.listdir(os.path.join(BASE_PATH, split, cls)))
        print(f"  {cls}: {count}")
        total += count
    print(f"  Total: {total}")

Cell 4: Compute Class Weights (handles imbalance)

In [ ]:
train_labels = []
for idx, cls in enumerate(CLASSES):
    folder = os.path.join(TRAIN_DIR, cls)
    train_labels.extend([idx] * len(os.listdir(folder)))

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=np.array(train_labels)
)
class_weight_dict = {0: class_weights[0],
                     1: class_weights[1],
                     2: class_weights[2]}
print("\nClass weights:", class_weight_dict)

Cell 5: Data Generators

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,       # brain is roughly symmetric
    rotation_range=10,          # small rotation
    zoom_range=0.05,            # very conservative zoom
    width_shift_range=0.05,
    height_shift_range=0.05,
    fill_mode='nearest'
)

# Validation and test generators — NO augmentation
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='rgb',
    class_mode='categorical',
    classes=CLASSES,
    shuffle=True
)

val_generator = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='rgb',
    class_mode='categorical',
    classes=CLASSES,
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='rgb',
    class_mode='categorical',
    classes=CLASSES,
    shuffle=False
)

print("\nClass indices:", train_generator.class_indices)

### MODEL 1: Simple CNN (replicates Gunawardena et al. 2017)

Cell 6: Build Simple CNN

In [ ]:
def build_simple_cnn(input_shape=(160, 160, 3), num_classes=3):
    model = models.Sequential([
        # Conv Block 1 (Unit 1: Convolution + ReLU)
        layers.Conv2D(32, (3, 3), activation='relu',
                      input_shape=input_shape, padding='same'),
        layers.BatchNormalization(),          # Unit 4: Batch Normalization

        # Conv Block 2
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),

        # Pooling (Unit 1: Downsampling)
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Dropout(0.25),                # Unit 2: Dropout Regularization

        # Conv Block 3
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),

        # Conv Block 4
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),

        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Dropout(0.25),

        # Fully Connected (Unit 1: Classification layer)
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),                 # Unit 2: Dropout
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

cnn_model = build_simple_cnn()
cnn_model.summary()

Cell 7: Compile Simple CNN

In [ ]:
cnn_model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

Cell 8: Callbacks

In [ ]:
cnn_callbacks = [
    # Unit 2: Early Stopping — prevents overfitting
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    # Reduce learning rate when stuck
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    # Save best model
    callbacks.ModelCheckpoint(
        '/content/drive/MyDrive/ADNI1/models/best_cnn.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

os.makedirs('/content/drive/MyDrive/ADNI1/models', exist_ok=True)

Cell 9: Train Simple CNN

In [ ]:
print("\nTraining Simple CNN...")
cnn_history = cnn_model.fit(
    train_generator,
    epochs=EPOCHS_CNN,
    validation_data=val_generator,
    class_weight=class_weight_dict,   # Unit 2: handles class imbalance
    callbacks=cnn_callbacks,
    verbose=1
)

Cell 10: Evaluate Simple CNN

In [ ]:
print("\nEvaluating Simple CNN on test set...")
cnn_test_loss, cnn_test_acc = cnn_model.evaluate(test_generator, verbose=0)
print(f"Test Accuracy: {cnn_test_acc*100:.2f}%")
print(f"Test Loss:     {cnn_test_loss:.4f}")

# Detailed classification report
test_generator.reset()
y_pred_probs = cnn_model.predict(test_generator, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_generator.classes

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=CLASSES))

Cell 11: Plot CNN Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(cnn_history.history['accuracy'],     label='Train')
axes[0].plot(cnn_history.history['val_accuracy'], label='Validation')
axes[0].set_title('Simple CNN — Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(cnn_history.history['loss'],     label='Train')
axes[1].plot(cnn_history.history['val_loss'], label='Validation')
axes[1].set_title('Simple CNN — Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/ADNI1/models/cnn_training_history.png', dpi=150)
plt.show()

Cell 12: CNN Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title('Simple CNN — Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/ADNI1/models/cnn_confusion_matrix.png', dpi=150)
plt.show()

### MODEL 2: ResNet50 Transfer Learning

In [ ]:
# Cell 1 - Imports
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from google.colab import drive
drive.mount('/content/drive')
print("GPU:", tf.config.list_physical_devices('GPU'))

In [ ]:
# Cell 2 - Copy data to local disk
if not os.path.exists("/content/processed"):
    import shutil
    print("Copying... please wait")
    shutil.copytree(
        "/content/drive/MyDrive/ADNI1/processed",
        "/content/processed"
    )
    print("✅ Done!")
else:
    print("✅ Already exists!")

Cell 13: Data Generators for ResNet50

In [ ]:
# ResNet50 needs its own preprocessing
resnet_train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    horizontal_flip=True,
    rotation_range=10,
    zoom_range=0.05,
    width_shift_range=0.05,
    height_shift_range=0.05,
    fill_mode='nearest'
)

resnet_val_test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

resnet_train_gen = resnet_train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='rgb',
    class_mode='categorical',
    classes=CLASSES,
    shuffle=True
)

resnet_val_gen = resnet_val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='rgb',
    class_mode='categorical',
    classes=CLASSES,
    shuffle=False
)

resnet_test_gen = resnet_val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='rgb',
    class_mode='categorical',
    classes=CLASSES,
    shuffle=False
)

Cell 14: Build ResNet50 Model

In [ ]:
def build_resnet50(input_shape=(160, 160, 3), num_classes=3):
    # Load ResNet50 pretrained on ImageNet — Unit 4: Transfer Learning
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,          # remove ImageNet classification head
        input_shape=input_shape
    )

    # Phase 1: Freeze all base layers — train only top layers first
    base_model.trainable = False

    inputs = tf.keras.Input(shape=input_shape)
    x = base_model(inputs, training=False)

    # Custom classification head
    x = layers.GlobalAveragePooling2D()(x)    # Unit 4: replaces Flatten
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)         # Unit 4: Batch Normalization
    x = layers.Dropout(0.5)(x)                # Unit 2: Dropout
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs)
    return model, base_model

resnet_model, base_model = build_resnet50()
resnet_model.summary()

Cell 15: Phase 1 — Train top layers only

In [ ]:
resnet_model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

resnet_callbacks_phase1 = [
    callbacks.EarlyStopping(
        monitor='val_loss', patience=5,
        restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=3, min_lr=1e-6, verbose=1
    )
]

print("\nPhase 1: Training top layers only...")
history_phase1 = resnet_model.fit(
    resnet_train_gen,
    epochs=10,
    validation_data=resnet_val_gen,
    class_weight=class_weight_dict,
    callbacks=resnet_callbacks_phase1,
    verbose=1
)

Cell 16: Phase 2 — Fine-tune last 30 layers

In [ ]:
# Unfreeze last 30 layers of ResNet50 for fine-tuning
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

print(f"\nPhase 2: Fine-tuning — trainable layers: "
      f"{sum(1 for l in resnet_model.layers if l.trainable)}")

# Lower learning rate for fine-tuning
resnet_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

resnet_callbacks_phase2 = [
    callbacks.EarlyStopping(
        monitor='val_loss', patience=7,
        restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=3, min_lr=1e-7, verbose=1
    ),
    callbacks.ModelCheckpoint(
        '/content/drive/MyDrive/ADNI1/models/best_resnet50.keras',
        monitor='val_accuracy',
        save_best_only=True, verbose=1
    )
]

print("Phase 2: Fine-tuning ResNet50...")
history_phase2 = resnet_model.fit(
    resnet_train_gen,
    epochs=EPOCHS_RES,
    validation_data=resnet_val_gen,
    class_weight=class_weight_dict,
    callbacks=resnet_callbacks_phase2,
    verbose=1
)


Cell 17: Evaluate ResNet50

In [ ]:
print("\nEvaluating ResNet50 on test set...")
res_test_loss, res_test_acc = resnet_model.evaluate(resnet_test_gen, verbose=0)
print(f"Test Accuracy: {res_test_acc*100:.2f}%")
print(f"Test Loss:     {res_test_loss:.4f}")

resnet_test_gen.reset()
y_pred_probs_res = resnet_model.predict(resnet_test_gen, verbose=0)
y_pred_res = np.argmax(y_pred_probs_res, axis=1)
y_true_res = resnet_test_gen.classes

print("\nClassification Report:")
print(classification_report(y_true_res, y_pred_res, target_names=CLASSES))

Cell 18: Plot ResNet50 Training History

In [ ]:
# Combine phase 1 and phase 2 history
combined_acc     = history_phase1.history['accuracy']     + history_phase2.history['accuracy']
combined_val_acc = history_phase1.history['val_accuracy'] + history_phase2.history['val_accuracy']
combined_loss    = history_phase1.history['loss']         + history_phase2.history['loss']
combined_val_loss= history_phase1.history['val_loss']     + history_phase2.history['val_loss']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(combined_acc,     label='Train')
axes[0].plot(combined_val_acc, label='Validation')
axes[0].axvline(x=len(history_phase1.history['accuracy']),
                color='red', linestyle='--', label='Fine-tuning starts')
axes[0].set_title('ResNet50 — Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(combined_loss,     label='Train')
axes[1].plot(combined_val_loss, label='Validation')
axes[1].axvline(x=len(history_phase1.history['loss']),
                color='red', linestyle='--', label='Fine-tuning starts')
axes[1].set_title('ResNet50 — Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/ADNI1/models/resnet_training_history.png', dpi=150)
plt.show()

Cell 19: ResNet50 Confusion Matrix

In [ ]:
cm_res = confusion_matrix(y_true_res, y_pred_res)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_res, annot=True, fmt='d', cmap='Greens',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title('ResNet50 — Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/ADNI1/models/resnet_confusion_matrix.png', dpi=150)
plt.show()

Cell 20: Final Comparison Table

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

def get_metrics(y_true, y_pred):
    acc  = accuracy_score(y_true, y_pred)
    sens = recall_score(y_true, y_pred, average='macro')  # sensitivity
    prec = precision_score(y_true, y_pred, average='macro')
    f1   = f1_score(y_true, y_pred, average='macro')
    return acc, sens, prec, f1

cnn_metrics = get_metrics(y_true, y_pred)
res_metrics = get_metrics(y_true_res, y_pred_res)

print("\n" + "="*65)
print(f"{'MODEL':<25} {'ACC':>8} {'SENS':>8} {'PREC':>8} {'F1':>8}")
print("="*65)
print(f"{'SVM (paper baseline)':<25} {'84.4%':>8} {'95.3%':>8} {'  -':>8} {'  -':>8}")
print(f"{'Simple CNN (ours)':<25} {cnn_metrics[0]*100:>7.1f}% {cnn_metrics[1]*100:>7.1f}% {cnn_metrics[2]*100:>7.1f}% {cnn_metrics[3]*100:>7.1f}%")
print(f"{'ResNet50 (ours)':<25} {res_metrics[0]*100:>7.1f}% {res_metrics[1]*100:>7.1f}% {res_metrics[2]*100:>7.1f}% {res_metrics[3]*100:>7.1f}%")
print("="*65)


In [ ]:
import psutil
ram = psutil.virtual_memory()
print(f"Total RAM: {ram.total / 1e9:.1f} GB")
print(f"Available RAM: {ram.available / 1e9:.1f} GB")

In [ ]:
import shutil
total, used, free = shutil.disk_usage("/content")
print(f"Total: {total / 1e9:.1f} GB")
print(f"Used:  {used / 1e9:.1f} GB")
print(f"Free:  {free / 1e9:.1f} GB")

In [ ]:
import shutil
import os

print("Copying dataset to local disk... (takes 5-10 mins)")

shutil.copytree(
    "/content/drive/MyDrive/ADNI1/processed",
    "/content/processed"
)

print("Done! Checking counts...")
for split in ["train", "val", "test"]:
    total = 0
    for cls in ["AD", "CN", "MCI"]:
        count = len(os.listdir(f"/content/processed/{split}/{cls}"))
        total += count
    print(f"{split}: {total} images")

In [ ]:
import os
print(os.path.exists("/content/processed"))
print(os.listdir("/content/processed") if os.path.exists("/content/processed") else "folder not found")

In [ ]:
import shutil
import os

print("Copying dataset to local disk... (takes 5-10 mins, please wait)")

shutil.copytree(
    "/content/drive/MyDrive/ADNI1/processed",
    "/content/processed"
)

print("✅ Done! Checking counts...")
for split in ["train", "val", "test"]:
    total = 0
    for cls in ["AD", "CN", "MCI"]:
        count = len(os.listdir(f"/content/processed/{split}/{cls}"))
        total += count
    print(f"{split}: {total} images")


# New Section

In [ ]:
# ============================================================
# CELL 1 — Imports & GPU check
# ============================================================
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from google.colab import drive

drive.mount('/content/drive')
print("GPU:", tf.config.list_physical_devices('GPU'))

In [ ]:
# ============================================================
# CELL 2 — Config  ← NEW, run this before anything else
# ============================================================
# Point directly at Drive — no copying needed
BASE_DIR  = "/content/drive/MyDrive/ADNI1/processed"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "val")
TEST_DIR  = os.path.join(BASE_DIR, "test")

IMG_SIZE   = (160, 160)
BATCH_SIZE = 32
EPOCHS_RES = 30
CLASSES    = ['CN', 'MCI', 'AD']   # must match your folder names exactly

MODEL_SAVE_DIR = "/content/drive/MyDrive/ADNI1/models"
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

print("Train:", TRAIN_DIR)
print("Val:  ", VAL_DIR)
print("Test: ", TEST_DIR)

In [ ]:
# ============================================================
# CELL 3 — Data generators
# ============================================================
resnet_train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    horizontal_flip=True,
    rotation_range=10,
    zoom_range=0.05,
    width_shift_range=0.05,
    height_shift_range=0.05,
    fill_mode='nearest'
)

resnet_val_test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

resnet_train_gen = resnet_train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='rgb',
    class_mode='categorical',
    classes=CLASSES,
    shuffle=True
)

resnet_val_gen = resnet_val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='rgb',
    class_mode='categorical',
    classes=CLASSES,
    shuffle=False
)

resnet_test_gen = resnet_val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='rgb',
    class_mode='categorical',
    classes=CLASSES,
    shuffle=False
)

# ── Class weights (computed right after generator, so .classes is available) ──
y_train_labels = resnet_train_gen.classes
class_weights  = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_labels),
    y=y_train_labels
)
class_weight_dict = dict(enumerate(class_weights))
print("Class weights:", class_weight_dict)

In [ ]:
# ============================================================
# CELL 4 — Build model
# ============================================================
def build_resnet50(input_shape=(160, 160, 3), num_classes=3):
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    base_model.trainable = False   # Phase 1: frozen

    inputs = tf.keras.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs)
    return model, base_model

resnet_model, base_model = build_resnet50()
resnet_model.summary()

In [ ]:
# ============================================================
# CELL 5 — Phase 1: train top layers only
# ============================================================
resnet_model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

resnet_callbacks_phase1 = [
    callbacks.EarlyStopping(
        monitor='val_loss', patience=5,
        restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=3, min_lr=1e-6, verbose=1
    ),
    # ← FIXED: checkpoint added to Phase 1 too
    callbacks.ModelCheckpoint(
        os.path.join(MODEL_SAVE_DIR, 'best_resnet50_phase1.keras'),
        monitor='val_accuracy',
        save_best_only=True, verbose=1
    )
]

print("Phase 1: Training top layers only...")
history_phase1 = resnet_model.fit(
    resnet_train_gen,
    epochs=10,
    validation_data=resnet_val_gen,
    class_weight=class_weight_dict,
    callbacks=resnet_callbacks_phase1,
    verbose=1
)

# Save history so a Colab crash doesn't lose it
import json
with open(os.path.join(MODEL_SAVE_DIR, 'history_phase1.json'), 'w') as f:
    json.dump(history_phase1.history, f)
print("Phase 1 history saved.")

In [ ]:
# ============================================================
# CELL 6 — Phase 2: fine-tune last 15 layers  (was 30 — reduced
#           to avoid overfitting on a small medical dataset)
# ============================================================
base_model.trainable = True
for layer in base_model.layers[:-15]:   # ← FIXED: 15 instead of 30
    layer.trainable = False

print(f"Phase 2: trainable layers: "
      f"{sum(1 for l in resnet_model.layers if l.trainable)}")

resnet_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

resnet_callbacks_phase2 = [
    callbacks.EarlyStopping(
        monitor='val_loss', patience=7,
        restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=3, min_lr=1e-7, verbose=1
    ),
    callbacks.ModelCheckpoint(
        os.path.join(MODEL_SAVE_DIR, 'best_resnet50.keras'),
        monitor='val_accuracy',
        save_best_only=True, verbose=1
    )
]

print("Phase 2: Fine-tuning ResNet50...")
history_phase2 = resnet_model.fit(
    resnet_train_gen,
    epochs=EPOCHS_RES,
    validation_data=resnet_val_gen,
    class_weight=class_weight_dict,
    callbacks=resnet_callbacks_phase2,
    verbose=1
)

with open(os.path.join(MODEL_SAVE_DIR, 'history_phase2.json'), 'w') as f:
    json.dump(history_phase2.history, f)
print("Phase 2 history saved.")

In [ ]:
# ============================================================
# CELL 7 — Evaluate on test set
# ============================================================
# ← FIXED: reset before evaluate AND before predict
resnet_test_gen.reset()
res_test_loss, res_test_acc = resnet_model.evaluate(resnet_test_gen, verbose=0)
print(f"Test Accuracy: {res_test_acc*100:.2f}%")
print(f"Test Loss:     {res_test_loss:.4f}")

resnet_test_gen.reset()   # ← reset again before predict
y_pred_probs_res = resnet_model.predict(resnet_test_gen, verbose=0)
y_pred_res = np.argmax(y_pred_probs_res, axis=1)
y_true_res = resnet_test_gen.classes

print("\nClassification Report:")
print(classification_report(y_true_res, y_pred_res, target_names=CLASSES))

In [ ]:
# ============================================================
# CELL 8 — Plot training history
# ============================================================
combined_acc      = history_phase1.history['accuracy']     + history_phase2.history['accuracy']
combined_val_acc  = history_phase1.history['val_accuracy'] + history_phase2.history['val_accuracy']
combined_loss     = history_phase1.history['loss']         + history_phase2.history['loss']
combined_val_loss = history_phase1.history['val_loss']     + history_phase2.history['val_loss']
split = len(history_phase1.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, train, val, title, ylabel in zip(
    axes,
    [combined_acc, combined_loss],
    [combined_val_acc, combined_val_loss],
    ['ResNet50 — Accuracy', 'ResNet50 — Loss'],
    ['Accuracy', 'Loss']
):
    ax.plot(train, label='Train')
    ax.plot(val,   label='Validation')
    ax.axvline(x=split, color='red', linestyle='--', label='Fine-tuning starts')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_SAVE_DIR, 'resnet_training_history.png'), dpi=150)
plt.show()

In [ ]:
# ============================================================
# CELL 9 — Confusion matrix
# ============================================================
cm_res = confusion_matrix(y_true_res, y_pred_res)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_res, annot=True, fmt='d', cmap='Greens',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title('ResNet50 — Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_SAVE_DIR, 'resnet_confusion_matrix.png'), dpi=150)
plt.show()

In [ ]:
import os
import random
import shutil

random.seed(42)

base = "/content/drive/MyDrive/ADNI1/processed"
classes = ["AD", "CN", "MCI"]

# Get all unique subject IDs across train/val/test
for cls in classes:
    all_files = []
    for split in ["train", "val", "test"]:
        folder = os.path.join(base, split, cls)
        files = os.listdir(folder)
        all_files.extend([(f, split) for f in files])

    # Extract unique subject IDs
    subject_ids = set()
    for f, _ in all_files:
        # baseline: ADNI_002_S_0816.nii_55.png → subject = 000_S_0000
        # month6:   m6_002_S_0816_55.png → subject = 000_S_0000
        if f.startswith("m6_"):
            sid = "_".join(f.replace("m6_", "").split("_")[:3])
        else:
            sid = "_".join(f.replace("ADNI_", "").split("_")[:3])
        subject_ids.add(sid)

    subject_ids = sorted(list(subject_ids))
    random.shuffle(subject_ids)
    n = len(subject_ids)
    train_sids = set(subject_ids[:int(0.70*n)])
    val_sids   = set(subject_ids[int(0.70*n):int(0.85*n)])
    test_sids  = set(subject_ids[int(0.85*n):])

    print(f"{cls}: {len(train_sids)} train / {len(val_sids)} val / {len(test_sids)} test subjects")

In [ ]:
import os
import random
import shutil

random.seed(42)

# ── Paths ─────────────────────────────────────────────────────
OLD_BASE = "/content/drive/MyDrive/ADNI1/processed"       # current messy split
NEW_BASE = "/content/drive/MyDrive/ADNI1/processed_clean" # new correct split
CLASSES  = ["AD", "CN", "MCI"]

# ── Step 1: Collect ALL files per class across all splits ─────
print("Step 1: Collecting all files...\n")

all_files = {cls: [] for cls in CLASSES}

for cls in CLASSES:
    for split in ["train", "val", "test"]:
        folder = os.path.join(OLD_BASE, split, cls)
        if not os.path.exists(folder):
            continue
        for f in os.listdir(folder):
            all_files[cls].append({
                "filename": f,
                "src_path": os.path.join(folder, f)
            })
    print(f"{cls}: {len(all_files[cls])} total slices")

# ── Step 2: Extract subject ID from filename ──────────────────
def get_subject_id(filename):
    """
    baseline: ADNI_002_S_0816.nii_55.png → 000_S_0000
    month6:   m6_002_S_0816_55.png       → 000_S_0000
    """
    if filename.startswith("m6_"):
        parts = filename.replace("m6_", "").split("_")
    else:
        parts = filename.replace("ADNI_", "").split("_")
    # Subject ID format is always XXX_S_XXXX
    for i in range(len(parts) - 2):
        if parts[i+1] == "S":
            return f"{parts[i]}_S_{parts[i+2]}"
    return None

# ── Step 3: Group files by subject ID ────────────────────────
print("\nStep 2: Grouping by subject ID...")

subject_files = {cls: {} for cls in CLASSES}

for cls in CLASSES:
    for item in all_files[cls]:
        sid = get_subject_id(item["filename"])
        if sid is None:
            print(f"  WARNING: Could not parse subject ID from {item['filename']}")
            continue
        if sid not in subject_files[cls]:
            subject_files[cls][sid] = []
        subject_files[cls][sid].append(item["src_path"])

    print(f"{cls}: {len(subject_files[cls])} unique subjects")

# ── Step 4: Split subjects 70/15/15 ──────────────────────────
print("\nStep 3: Splitting subjects 70/15/15...")

split_subjects = {}
for cls in CLASSES:
    sids = sorted(list(subject_files[cls].keys()))
    random.shuffle(sids)
    n         = len(sids)
    train_end = int(0.70 * n)
    val_end   = int(0.85 * n)

    split_subjects[cls] = {
        "train": sids[:train_end],
        "val":   sids[train_end:val_end],
        "test":  sids[val_end:]
    }

    print(f"{cls}: {len(split_subjects[cls]['train'])} train / "
          f"{len(split_subjects[cls]['val'])} val / "
          f"{len(split_subjects[cls]['test'])} test subjects")

# ── Step 5: Copy files to new structure ──────────────────────
print("\nStep 4: Building clean dataset (this may take a while)...")

for cls in CLASSES:
    for split_type, sids in split_subjects[cls].items():
        dst_folder = os.path.join(NEW_BASE, split_type, cls)
        os.makedirs(dst_folder, exist_ok=True)

        for sid in sids:
            for src_path in subject_files[cls][sid]:
                filename = os.path.basename(src_path)
                dst_path = os.path.join(dst_folder, filename)
                shutil.copy2(src_path, dst_path)

    print(f"  {cls} done!")

# ── Step 6: Verify final counts ───────────────────────────────
print("\n── Final Clean Dataset Counts ──")
for split in ["train", "val", "test"]:
    print(f"\n{split.upper()}")
    total = 0
    for cls in CLASSES:
        folder = os.path.join(NEW_BASE, split, cls)
        count  = len(os.listdir(folder))
        print(f"  {cls}: {count}")
        total += count
    print(f"  Total: {total}")

print(f"\n✅ Clean dataset saved to: {NEW_BASE}")
print("Update your BASE_PATH to this new folder and retrain!")

In [ ]:
"/content/drive/MyDrive/ADNI1"

In [ ]:
import os
from PIL import Image
import numpy as np
from collections import defaultdict

DATA_ROOT = "/content/drive/MyDrive/ADNI1/processed"

def audit_dataset(data_root):
    for split in ["train", "val", "test"]:
        split_path = os.path.join(data_root, split)
        if not os.path.exists(split_path):
            print(f"❌ Missing split folder: {split}")
            continue

        print(f"\n{'='*50}")
        print(f"  {split.upper()}")
        print(f"{'='*50}")

        classes = sorted(os.listdir(split_path))
        print(f"Classes found: {classes}")

        total = 0
        for cls in classes:
            cls_path = os.path.join(split_path, cls)
            if not os.path.isdir(cls_path):
                continue

            files = os.listdir(cls_path)
            images = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            non_images = [f for f in files if not f.lower().endswith(('.png', '.jpg', '.jpeg'))]

            print(f"\n  [{cls}] → {len(images)} images", end="")
            if non_images:
                print(f"  ⚠️  {len(non_images)} non-image files: {non_images[:5]}")
            else:
                print()

            # Sample 5 images and check them
            corrupt = 0
            sizes = defaultdict(int)
            modes = defaultdict(int)
            for fname in images[:200]:  # check first 200 per class
                fpath = os.path.join(cls_path, fname)
                try:
                    img = Image.open(fpath)
                    sizes[img.size] += 1
                    modes[img.mode] += 1
                    img.verify()
                except Exception as e:
                    corrupt += 1

            print(f"    Sizes (sample): {dict(list(sizes.items())[:3])}")
            print(f"    Modes (color/grayscale): {dict(modes)}")
            if corrupt:
                print(f"    ❌ Corrupt images found: {corrupt}")
            else:
                print(f"    ✅ No corrupt images in sample")

            total += len(images)

        print(f"\n  Total in {split}: {total}")

audit_dataset(DATA_ROOT)

In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from collections import defaultdict

DATA_ROOT = "/content/drive/MyDrive/ADNI1/processed"
SPLITS = ["train", "val", "test"]

# Thresholds — tune these if needed
BLACK_PIXEL_THRESHOLD = 0.85   # flag if >85% of pixels are near-black
BRIGHT_PIXEL_THRESHOLD = 0.95  # flag if >95% of pixels are near-white
NEAR_BLACK_VALUE = 10          # pixel value considered "black" (0-255)

def analyze_slice(fpath):
    img = Image.open(fpath).convert("L")  # grayscale
    arr = np.array(img)
    total = arr.size

    black_ratio  = np.sum(arr < NEAR_BLACK_VALUE) / total
    white_ratio  = np.sum(arr > 245) / total
    mean_val     = arr.mean()
    std_val      = arr.std()

    flags = []
    if black_ratio > BLACK_PIXEL_THRESHOLD:
        flags.append(f"mostly_black ({black_ratio:.0%} black pixels)")
    if white_ratio > BRIGHT_PIXEL_THRESHOLD:
        flags.append(f"mostly_white ({white_ratio:.0%} white pixels)")
    if std_val < 5:
        flags.append(f"no_variance (std={std_val:.1f})")

    return {
        "path": fpath,
        "black_ratio": black_ratio,
        "white_ratio": white_ratio,
        "mean": mean_val,
        "std": std_val,
        "flags": flags,
        "bad": len(flags) > 0
    }

# ── Run audit ────────────────────────────────────────────────────────────────
all_bad = []
summary = defaultdict(lambda: defaultdict(int))

for split in SPLITS:
    split_path = os.path.join(DATA_ROOT, split)
    if not os.path.exists(split_path):
        continue
    for cls in sorted(os.listdir(split_path)):
        cls_path = os.path.join(split_path, cls)
        if not os.path.isdir(cls_path):
            continue
        files = [f for f in os.listdir(cls_path) if f.lower().endswith(('.png','.jpg','.jpeg'))]
        for fname in files:
            result = analyze_slice(os.path.join(cls_path, fname))
            summary[split][cls] += 1
            if result["bad"]:
                all_bad.append((split, cls, result))

# ── Print report ─────────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print(f"  BAD SLICE REPORT")
print(f"{'='*55}")
print(f"Total bad slices found: {len(all_bad)}\n")

by_split_class = defaultdict(list)
for split, cls, result in all_bad:
    by_split_class[(split, cls)].append(result)

for (split, cls), results in sorted(by_split_class.items()):
    total_in_class = summary[split][cls]
    pct = 100 * len(results) / total_in_class
    print(f"  {split}/{cls}: {len(results)} bad / {total_in_class} total ({pct:.1f}%)")
    flag_counts = defaultdict(int)
    for r in results:
        for f in r["flags"]:
            flag_counts[f.split("(")[0].strip()] += 1
    for flag, count in flag_counts.items():
        print(f"    └─ {flag}: {count}")

# ── Visualize worst offenders ─────────────────────────────────────────────────
if all_bad:
    print(f"\nShowing sample bad slices...")
    sample = all_bad[:12]
    fig, axes = plt.subplots(3, 4, figsize=(14, 10))
    axes = axes.flatten()
    for i, (split, cls, result) in enumerate(sample):
        img = Image.open(result["path"]).convert("L")
        axes[i].imshow(img, cmap="gray")
        axes[i].set_title(f"{split}/{cls}\n{', '.join(result['flags'])}", fontsize=7)
        axes[i].axis("off")
    for j in range(i+1, len(axes)):
        axes[j].axis("off")
    plt.tight_layout()
    plt.savefig("bad_slices_sample.png", dpi=100)
    plt.show()
    print("Saved → bad_slices_sample.png")
else:
    print("\n✅ No obviously bad slices found!")

In [ ]:
import os
from collections import defaultdict

DATA_ROOT = "/content/drive/MyDrive/ADNI1/processed"

# This works if your filenames encode the subject/scan ID
# Common ADNI naming: XXXXX_S_XXXX_scandate_sliceN.png
# We strip the slice number to get the parent scan ID

def get_scan_id(filename):
    name = os.path.splitext(filename)[0]  # remove .png
    # Remove trailing .nii if present (for ADNI_ format)
    name = name.replace(".nii", "")
    # Strip last _number (slice index)
    parts = name.rsplit("_", 1)
    return parts[0] if parts[-1].isdigit() else name

# Collect all scan IDs per split
split_scans = defaultdict(set)
split_files = defaultdict(int)

for split in ["train", "val", "test"]:
    split_path = os.path.join(DATA_ROOT, split)
    for cls in os.listdir(split_path):
        cls_path = os.path.join(split_path, cls)
        if not os.path.isdir(cls_path):
            continue
        for fname in os.listdir(cls_path):
            if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                scan_id = get_scan_id(fname)
                split_scans[split].add(scan_id)
                split_files[split] += 1

# Print scan counts
print("=== Unique Scans per Split ===")
for split in ["train", "val", "test"]:
    print(f"  {split}: {len(split_scans[split])} unique scans, {split_files[split]} total slices")

# Check for overlap
train_val_overlap  = split_scans["train"] & split_scans["val"]
train_test_overlap = split_scans["train"] & split_scans["test"]
val_test_overlap   = split_scans["val"]   & split_scans["test"]

print("\n=== Leakage Check ===")
if train_test_overlap:
    print(f"  ❌ Train/Test overlap: {len(train_test_overlap)} scans LEAKED")
    print(f"     Sample: {list(train_test_overlap)[:5]}")
else:
    print(f"  ✅ Train/Test: No overlap")

if train_val_overlap:
    print(f"  ⚠️  Train/Val overlap: {len(train_val_overlap)} scans")
else:
    print(f"  ✅ Train/Val: No overlap")

if val_test_overlap:
    print(f"  ⚠️  Val/Test overlap: {len(val_test_overlap)} scans")
else:
    print(f"  ✅ Val/Test: No overlap")

# Also print a few filenames so you can verify the ID extraction is correct
print("\n=== Sample Filename → Scan ID (verify this looks right) ===")
for split in ["train"]:
    split_path = os.path.join(DATA_ROOT, split)
    for cls in os.listdir(split_path):
        cls_path = os.path.join(split_path, cls)
        if not os.path.isdir(cls_path): continue
        files = [f for f in os.listdir(cls_path) if f.endswith('.png')][:3]
        for f in files:
            print(f"  {f}  →  {get_scan_id(f)}")
        break

In [ ]:
import os, shutil, random
from collections import defaultdict

DATA_ROOT  = "/content/drive/MyDrive/ADNI1/processed"
OUTPUT_ROOT = "/content/drive/MyDrive/ADNI1/processed_fixed"  # new clean dataset
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15
SEED        = 42
random.seed(SEED)

def get_scan_id(filename):
    name = os.path.splitext(filename)[0].replace(".nii", "")
    parts = name.rsplit("_", 1)
    return parts[0] if parts[-1].isdigit() else name

# ── Collect all files grouped by (class, scan_id) ───────────────────────────
class_scan_files = defaultdict(lambda: defaultdict(list))

for split in ["train", "val", "test"]:
    split_path = os.path.join(DATA_ROOT, split)
    for cls in os.listdir(split_path):
        cls_path = os.path.join(split_path, cls)
        if not os.path.isdir(cls_path):
            continue
        for fname in os.listdir(cls_path):
            if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                scan_id = get_scan_id(fname)
                class_scan_files[cls][scan_id].append(
                    os.path.join(cls_path, fname)
                )

# ── Split at scan level, copy files ─────────────────────────────────────────
print("Rebuilding splits at scan level...\n")

for cls, scan_dict in class_scan_files.items():
    scan_ids = list(scan_dict.keys())
    random.shuffle(scan_ids)

    n = len(scan_ids)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)

    splits = {
        "train": scan_ids[:n_train],
        "val":   scan_ids[n_train:n_train + n_val],
        "test":  scan_ids[n_train + n_val:]
    }

    print(f"[{cls}] Total scans: {n} → "
          f"train: {len(splits['train'])} | "
          f"val: {len(splits['val'])} | "
          f"test: {len(splits['test'])}")

    for split_name, ids in splits.items():
        out_dir = os.path.join(OUTPUT_ROOT, split_name, cls)
        os.makedirs(out_dir, exist_ok=True)
        for scan_id in ids:
            for fpath in scan_dict[scan_id]:
                shutil.copy(fpath, os.path.join(out_dir, os.path.basename(fpath)))

print("\n✅ Done. Clean dataset saved to:", OUTPUT_ROOT)

# ── Verify no leakage ────────────────────────────────────────────────────────
from collections import defaultdict as dd
split_scans = dd(set)
for split in ["train", "val", "test"]:
    for cls in os.listdir(os.path.join(OUTPUT_ROOT, split)):
        cls_path = os.path.join(OUTPUT_ROOT, split, cls)
        if not os.path.isdir(cls_path): continue
        for fname in os.listdir(cls_path):
            split_scans[split].add(get_scan_id(fname))

print("\n=== Leakage Check on Fixed Dataset ===")
print(f"  Train/Test overlap: {len(split_scans['train'] & split_scans['test'])} scans")
print(f"  Train/Val overlap:  {len(split_scans['train'] & split_scans['val'])} scans")
print(f"  Val/Test overlap:   {len(split_scans['val']   & split_scans['test'])} scans")

In [ ]:
import os, torch, torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# ── Config ──────────────────────────────────────────────────────────────────
DATA_ROOT  = "/content/drive/MyDrive/ADNI1/processed_fixed"   # folder with train/val/test subfolders
BATCH      = 32
EPOCHS     = 10
LR         = 1e-4
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASSES    = ["AD", "CN"]              # drop MCI entirely

# ── Transforms ──────────────────────────────────────────────────────────────
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

def filter_dataset(dataset, keep_classes):
    """Keep only samples belonging to keep_classes."""
    class_to_idx = dataset.class_to_idx
    keep_idx = {class_to_idx[c] for c in keep_classes if c in class_to_idx}
    indices = [i for i, (_, label) in enumerate(dataset.samples) if label in keep_idx]
    subset = Subset(dataset, indices)
    return subset

# ── Datasets ────────────────────────────────────────────────────────────────
full_train = datasets.ImageFolder(os.path.join(DATA_ROOT, "train"), transform=train_tf)
full_val   = datasets.ImageFolder(os.path.join(DATA_ROOT, "val"),   transform=val_tf)
full_test  = datasets.ImageFolder(os.path.join(DATA_ROOT, "test"),  transform=val_tf)

train_ds = filter_dataset(full_train, CLASSES)
val_ds   = filter_dataset(full_val,   CLASSES)
test_ds  = filter_dataset(full_test,  CLASSES)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=2)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

# ── Model (ResNet50, pretrained) ─────────────────────────────────────────────
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 2)   # binary output
model = model.to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

# ── Training ─────────────────────────────────────────────────────────────────
for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (out.argmax(1) == labels).sum().item()
        total += labels.size(0)

    # Validation
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out = model(imgs)
            val_correct += (out.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    scheduler.step()
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.3f} "
          f"| Train Acc: {100*correct/total:.1f}% | Val Acc: {100*val_correct/val_total:.1f}%")

# ── Test Evaluation ───────────────────────────────────────────────────────────
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)
        preds = model(imgs).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print("\n=== Test Results ===")
print(classification_report(all_labels, all_preds, target_names=CLASSES))
print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))